In [ ]:

import math
from dataclasses import dataclass
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


# -----------------------------
# Config
# -----------------------------
@dataclass
class DiffusionConfig:
    """Configuration for diffusion training and sampling."""
    image_size: int = 28
    channels: int = 1
    batch_size: int = 128
    epochs: int = 5
    lr: float = 2e-4
    beta_min: float = 0.1
    beta_max: float = 20.0
    num_steps: int = 1000
    num_sample_grid: int = 16
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Helpers
# -----------------------------
def sinusoidal_embedding(t: torch.Tensor, dim: int = 128) -> torch.Tensor:
    """
    Build sinusoidal time embeddings.

    Mathematical form:
    $$\text{emb}(t) = [\sin(\omega_1 t), \dots, \sin(\omega_k t), \cos(\omega_1 t), \dots, \cos(\omega_k t)]$$

    Args:
        t: Normalized time tensor of shape [B].
        dim: Embedding dimension.

    Returns:
        Tensor of shape [B, dim].
    """
    half = dim // 2
    device = t.device
    freqs = torch.exp(
        torch.linspace(math.log(1.0), math.log(10000.0), half, device=device)
    )
    angles = t[:, None] * freqs[None, :]
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


# -----------------------------
# Model
# -----------------------------
class Block(nn.Module):
    """Residual convolution block with time conditioning."""
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.act(self.conv2(h))
        return h + self.skip(x)


class SimpleScoreNet(nn.Module):
    """
    Small score network for image diffusion.

    Input:
        x: Noisy image tensor of shape [B, C, H, W].
        t: Normalized time tensor of shape [B].

    Output:
        Tensor with the same shape as x.
    """
    def __init__(self, in_channels: int = 1, base_channels: int = 64, time_dim: int = 128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim),
        )

        self.in_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.block1 = Block(base_channels, base_channels, time_dim)
        self.down = nn.Conv2d(base_channels, base_channels * 2, 4, stride=2, padding=1)
        self.block2 = Block(base_channels * 2, base_channels * 2, time_dim)
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1)
        self.block3 = Block(base_channels, base_channels, time_dim)
        self.out = nn.Conv2d(base_channels, in_channels, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = sinusoidal_embedding(t, self.time_dim)
        t_emb = self.time_mlp(t_emb)

        x = self.in_conv(x)
        x = self.block1(x, t_emb)
        x = self.down(x)
        x = self.block2(x, t_emb)
        x = self.up(x)
        x = self.block3(x, t_emb)
        return self.out(x)

class Diffusion:
    """
    Diffusion process utilities.

    Stored state is limited to init parameters only:
    - beta schedule bounds
    - number of reverse steps
    - target device

    """
    def __init__(self, beta_min: float = 0.1, beta_max: float = 20.0, num_steps: int = 1000, device: str = "cpu"):
        self.beta_min = beta_min
        self.beta_max = beta_max
        self.num_steps = num_steps
        self.device = device

    def beta(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the linear beta schedule.

        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            Beta values for each time entry.
        """
        return self.beta_min + t * (self.beta_max - self.beta_min)

    def alpha_bar(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the cumulative signal retention.
        
        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            alpha_bar(t).
        """
        return torch.exp(
            -0.5 * (self.beta_min * t + 0.5 * (self.beta_max - self.beta_min) * t**2)
        )

    def sigma(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the forward noise scale.

        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            sigma(t).
        """
        return torch.sqrt(1.0 - self.alpha_bar(t).clamp(max=0.999999))

    def q_sample(
        self,
        x0: torch.Tensor,
        t: torch.Tensor,
        noise: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Apply the forward diffusion process.

        Args:
            x0: Clean image batch.
            t: Normalized time tensor.
            noise: Optional Gaussian noise tensor.

        Returns:
            Tuple of (noisy images, noise used, sigma values).
        """
        
        # If noise is not provided, sample standard Gaussian noise.
        if noise is None:
            noise = torch.randn_like(x0)

        # Compute the alpha_bar and sigma values for the given time tensor.
        alpha_bar = self.alpha_bar(t)[:, None, None, None]
        sigma = self.sigma(t)[:, None, None, None]
        
        # sampling step with gaussian distribution
        xt = torch.sqrt(alpha_bar) * x0 + sigma * noise
        
        # return parameters needed for loss computation
        return xt, noise, sigma

    @torch.no_grad()
    def reverse_sample(self, score_model: nn.Module, n: int = 16) -> torch.Tensor:
        """
        Generate images by running the reverse diffusion process.

        Args:
            score_model: Trained score model.
            n: Number of images to generate.

        Returns:
            Generated image batch of shape [n, C, H, W].
        """
        # setup
        score_model.eval()
        device = torch.device(self.device)
        
        # sample gaussian noise as initial state
        x = torch.randn(n, self.channels, self.image_size, self.image_size, device=device)
        
        # create time steps for reverse process
        steps = torch.linspace(1.0, 1e-3, self.num_steps, device=device) # avoiding zero
        dt = 1.0 / self.num_steps
        
        # sample the reverse process iteratively
        for t in steps:
            # precomputation
            t_batch = torch.full((n,), t, device=device)
            beta_t = self.beta(t_batch)[:, None, None, None]
            score = score_model(x, t_batch)

            # update step according to the reverse SDE
            drift = 0.5 * beta_t * x + beta_t * score
            noise = torch.randn_like(x)
            x = x + drift * dt + torch.sqrt(beta_t * dt) * noise

        return x.clamp(-1, 1)

    @property
    def image_size(self) -> int:
        return getattr(self, "_image_size", 28)

    @property
    def channels(self) -> int:
        return getattr(self, "_channels", 1)

    @channels.setter
    def channels(self, value: int) -> None:
        self._channels = value

    @image_size.setter
    def image_size(self, value: int) -> None:
        self._image_size = value

# -----------------------------
# Loss
# -----------------------------
def diffusion_loss(
    diffusion: Diffusion,
    model: nn.Module,
    x0: torch.Tensor,
) -> torch.Tensor:
    """
    Compute the denoising score matching loss.
    
    Args:
        cfg: Diffusion configuration.
        diffusion: Diffusion process helper.
        model: Score model taking (x, t).
        x0: Clean image batch.

    Returns:
        Scalar loss tensor.
    """
    t = torch.rand(x0.shape[0], device=x0.device).clamp(1e-5, 1.0)
    xt, noise, sigma = diffusion.q_sample(x0, t)
    pred = model(xt, t)
    target = -noise / sigma
    return ((sigma ** 2) * (pred - target).pow(2)).mean()

# -----------------------------
# Training
# -----------------------------
def train_diffusion_model(
    cfg: DiffusionConfig,
    diffusion: Diffusion,
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> Dict[str, Any]:
    """
    Train the diffusion model on a dataset.

    Args:
        cfg: Diffusion configuration.
        diffusion: Diffusion helper object.
        model: Score model to train.
        loader: DataLoader yielding image batches.
        optimizer: Optimizer used for parameter updates.

    Returns:
        Dictionary with the trained model and loss history.
    """
    history = []

    model.train()
    for epoch in range(cfg.epochs):
        running_loss = 0.0

        for x, _ in loader:
            x = x.to(cfg.device)

            optimizer.zero_grad(set_to_none=True)
            loss = diffusion_loss( diffusion, model, x)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(loader)
        history.append(avg_loss)
        print(f"epoch {epoch + 1:02d}/{cfg.epochs} | loss {avg_loss:.4f}")

import math
from dataclasses import dataclass
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


# -----------------------------
# Config
# -----------------------------
@dataclass
class DiffusionConfig:
    """Configuration for diffusion training and sampling."""
    image_size: int = 28
    channels: int = 1
    batch_size: int = 128
    epochs: int = 5
    lr: float = 2e-4
    beta_min: float = 0.1
    beta_max: float = 20.0
    num_steps: int = 1000
    num_sample_grid: int = 16
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Helpers
# -----------------------------
def sinusoidal_embedding(t: torch.Tensor, dim: int = 128) -> torch.Tensor:
    """
    Build sinusoidal time embeddings.

    Mathematical form:
    $$\text{emb}(t) = [\sin(\omega_1 t), \dots, \sin(\omega_k t), \cos(\omega_1 t), \dots, \cos(\omega_k t)]$$

    Args:
        t: Normalized time tensor of shape [B].
        dim: Embedding dimension.

    Returns:
        Tensor of shape [B, dim].
    """
    half = dim // 2
    device = t.device
    freqs = torch.exp(
        torch.linspace(math.log(1.0), math.log(10000.0), half, device=device)
    )
    angles = t[:, None] * freqs[None, :]
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


# -----------------------------
# Model
# -----------------------------
class Block(nn.Module):
    """Residual convolution block with time conditioning."""
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.act(self.conv2(h))
        return h + self.skip(x)


class SimpleScoreNet(nn.Module):
    """
    Small score network for image diffusion.

    Input:
        x: Noisy image tensor of shape [B, C, H, W].
        t: Normalized time tensor of shape [B].

    Output:
        Tensor with the same shape as x.
    """
    def __init__(self, in_channels: int = 1, base_channels: int = 64, time_dim: int = 128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim),
        )

        self.in_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.block1 = Block(base_channels, base_channels, time_dim)
        self.down = nn.Conv2d(base_channels, base_channels * 2, 4, stride=2, padding=1)
        self.block2 = Block(base_channels * 2, base_channels * 2, time_dim)
        self.up = nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1)
        self.block3 = Block(base_channels, base_channels, time_dim)
        self.out = nn.Conv2d(base_channels, in_channels, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = sinusoidal_embedding(t, self.time_dim)
        t_emb = self.time_mlp(t_emb)

        x = self.in_conv(x)
        x = self.block1(x, t_emb)
        x = self.down(x)
        x = self.block2(x, t_emb)
        x = self.up(x)
        x = self.block3(x, t_emb)
        return self.out(x)

class Diffusion:
    """
    Diffusion process utilities.

    Stored state is limited to init parameters only:
    - beta schedule bounds
    - number of reverse steps
    - target device

    """
    def __init__(self, beta_min: float = 0.1, beta_max: float = 20.0, num_steps: int = 1000, device: str = "cpu"):
        self.beta_min = beta_min
        self.beta_max = beta_max
        self.num_steps = num_steps
        self.device = device

    def beta(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the linear beta schedule.

        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            Beta values for each time entry.
        """
        return self.beta_min + t * (self.beta_max - self.beta_min)

    def alpha_bar(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the cumulative signal retention.
        
        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            alpha_bar(t).
        """
        return torch.exp(
            -0.5 * (self.beta_min * t + 0.5 * (self.beta_max - self.beta_min) * t**2)
        )

    def sigma(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute the forward noise scale.

        Args:
            t: Normalized time tensor in [0, 1].

        Returns:
            sigma(t).
        """
        return torch.sqrt(1.0 - self.alpha_bar(t).clamp(max=0.999999))

    def q_sample(
        self,
        x0: torch.Tensor,
        t: torch.Tensor,
        noise: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Apply the forward diffusion process.

        Args:
            x0: Clean image batch.
            t: Normalized time tensor.
            noise: Optional Gaussian noise tensor.

        Returns:
            Tuple of (noisy images, noise used, sigma values).
        """
        
        # If noise is not provided, sample standard Gaussian noise.
        if noise is None:
            noise = torch.randn_like(x0)

        # Compute the alpha_bar and sigma values for the given time tensor.
        alpha_bar = self.alpha_bar(t)[:, None, None, None]
        sigma = self.sigma(t)[:, None, None, None]
        
        # sampling step with gaussian distribution
        xt = torch.sqrt(alpha_bar) * x0 + sigma * noise
        
        # return parameters needed for loss computation
        return xt, noise, sigma

    @torch.no_grad()
    def reverse_sample(self, score_model: nn.Module, n: int = 16) -> torch.Tensor:
        """
        Generate images by running the reverse diffusion process.

        Args:
            score_model: Trained score model.
            n: Number of images to generate.

        Returns:
            Generated image batch of shape [n, C, H, W].
        """
        # setup
        score_model.eval()
        device = torch.device(self.device)
        
        # sample gaussian noise as initial state
        x = torch.randn(n, self.channels, self.image_size, self.image_size, device=device)
        
        # create time steps for reverse process
        steps = torch.linspace(1.0, 1e-3, self.num_steps, device=device) # avoiding zero
        dt = 1.0 / self.num_steps
        
        # sample the reverse process iteratively
        for t in steps:
            # precomputation
            t_batch = torch.full((n,), t, device=device)
            beta_t = self.beta(t_batch)[:, None, None, None]
            score = score_model(x, t_batch)

            # update step according to the reverse SDE
            drift = 0.5 * beta_t * x + beta_t * score
            noise = torch.randn_like(x)
            x = x + drift * dt + torch.sqrt(beta_t * dt) * noise

        return x.clamp(-1, 1)

    @property
    def image_size(self) -> int:
        return getattr(self, "_image_size", 28)

    @property
    def channels(self) -> int:
        return getattr(self, "_channels", 1)

    @channels.setter
    def channels(self, value: int) -> None:
        self._channels = value

    @image_size.setter
    def image_size(self, value: int) -> None:
        self._image_size = value

# -----------------------------
# Loss
# -----------------------------
def diffusion_loss(
    diffusion: Diffusion,
    model: nn.Module,
    x0: torch.Tensor,
) -> torch.Tensor:
    """
    Compute the denoising score matching loss.
    
    Args:
        cfg: Diffusion configuration.
        diffusion: Diffusion process helper.
        model: Score model taking (x, t).
        x0: Clean image batch.

    Returns:
        Scalar loss tensor.
    """
    t = torch.rand(x0.shape[0], device=x0.device).clamp(1e-5, 1.0)
    xt, noise, sigma = diffusion.q_sample(x0, t)
    pred = model(xt, t)
    target = -noise / sigma
    return ((sigma ** 2) * (pred - target).pow(2)).mean()

# -----------------------------
# Training
# -----------------------------
def train_diffusion_model(
    cfg: DiffusionConfig,
    diffusion: Diffusion,
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> Dict[str, Any]:
    """
    Train the diffusion model on a dataset.

    Args:
        cfg: Diffusion configuration.
        diffusion: Diffusion helper object.
        model: Score model to train.
        loader: DataLoader yielding image batches.
        optimizer: Optimizer used for parameter updates.

    Returns:
        Dictionary with the trained model and loss history.
    """
    history = []

    model.train()
    for epoch in range(cfg.epochs):
        running_loss = 0.0

        for x, _ in loader:
            x = x.to(cfg.device)

            optimizer.zero_grad(set_to_none=True)
            loss = diffusion_loss( diffusion, model, x)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(loader)
        history.append(avg_loss)
        print(f"epoch {epoch + 1:02d}/{cfg.epochs} | loss {avg_loss:.4f}")
        
        # Save the model
        #model_path = f"model_epoch_{epoch + 1:02d}.pt"
        #torch.save(model.state_dict(), model_path)
        #print(f"Saved network to {model_path}")
        
        # Evaluate model by generating images
        import matplotlib.pyplot as plt
        print(f"Evaluating epoch {epoch + 1}...")
        
        # Store training mode state and switch to eval
        training_state = model.training
        
        samples = generate_images(cfg, diffusion, model, num_images=16)
        fig, axes = plt.subplots(4, 4, figsize=(4, 4))
        for i, ax in enumerate(axes.flatten()):
            img = (samples[i].cpu().squeeze() + 1) / 2 # denormalize
            ax.imshow(img.clamp(0, 1), cmap='gray')
            ax.axis('off')
        plt.suptitle(f"Epoch {epoch + 1} Evaluation")
        plt.tight_layout()
        plt.show()
        
        # Restore training state
        model.train(training_state)

    return {"model": model, "history": history}


@torch.no_grad()
def generate_images(
    cfg: DiffusionConfig,
    diffusion: Diffusion,
    model: nn.Module,
    num_images: int = 16,
) -> torch.Tensor:
    """
    Generate images from a trained diffusion model.

    Args:
        cfg: Diffusion configuration.
        diffusion: Diffusion helper object.
        model: Trained score model.
        num_images: Number of images to generate.

    Returns:
        Generated image batch.
    """
    diffusion.channels = cfg.channels
    diffusion.image_size = cfg.image_size
    return diffusion.reverse_sample(model, n=num_images)
 


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt





# ==========================================
# SMART PARAMETER CONFIGURATION
# ==========================================
# Data Params
IMAGE_SIZE = 28
CHANNELS = 1
BATCH_SIZE = 128      # 128 balances gradient stability and memory for a small dataset.
TARGET_DIGITS = [6, 7] # The specific MNIST digits we want to model.

# Model Params
BASE_DIM = 84         # Base channel dimension for the medium U-Net.
TIME_DIM = 128        # Explicit time embedding dimension for better step conditioning.

# Training Params
EPOCHS = 10        # ~7-10 epochs handles this digit subset nicely.
LEARNING_RATE = 1e-3  # 1e-3 is ideal for Adam on this scale.

# Diffusion Process Params
BETA_MIN = 0.01        # Standard continuous-time lower bound for noise.
BETA_MAX = 20.0       # Standard continuous-time upper bound to reach pure Gaussian.
NUM_STEPS = 1000      # 1000 steps ensure smooth transitions during the reverse ODE/SDE.
NUM_GENERATIONS = 16  # For the 4x4 grid plot.
# ==========================================

# 2. Define the Custom Medium-Sized U-Net Network
class ConvBlock(nn.Module):
    """A basic Convolutional block with Time conditioning and a residual connection."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x, t_emb):
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[..., None, None]
        h = self.act(self.conv2(h))
        return h + self.skip(x)

class UNetMedium(nn.Module):
    """
    A medium-sized U-Net with skip connections, downsampling, bottleneck, and upsampling.
    """
    def __init__(self, in_channels=CHANNELS, base_dim=BASE_DIM, time_dim=TIME_DIM):
        super().__init__()
        self.time_dim = time_dim
        
        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.GELU(),
            nn.Linear(time_dim * 4, time_dim)
        )
        
        self.inc = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        
        # Downsampling path (spatial dim halved with maxpool, channels doubled in block)
        self.down1 = ConvBlock(base_dim, base_dim, time_dim)
        self.down2 = ConvBlock(base_dim, base_dim * 2, time_dim)
        self.down3 = ConvBlock(base_dim * 2, base_dim * 4, time_dim)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck (Mid)
        self.mid = ConvBlock(base_dim * 4, base_dim * 4, time_dim)
        
        # Upsampling path
        self.up1 = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 2, stride=2)
        self.up_block1 = ConvBlock(base_dim * 4, base_dim * 2, time_dim) # in_ch = base*2(up) + base*2(skip) = base*4
        
        self.up2 = nn.ConvTranspose2d(base_dim * 2, base_dim, 2, stride=2)
        self.up_block2 = ConvBlock(base_dim * 2, base_dim, time_dim) # in_ch = base(up) + base(skip) = base*2
        
        self.outc = nn.Conv2d(base_dim, in_channels, 3, padding=1)

    def forward(self, x, t):
        # Time embeddings
        t_emb = sinusoidal_embedding(t, self.time_dim) 
        t_emb = self.time_mlp(t_emb)
        
        # Initial 
        x0 = self.inc(x) # 28x28
        
        # Down 1
        d1 = self.down1(x0, t_emb) # 28x28
        p1 = self.pool(d1)         # 14x14
        
        # Down 2
        d2 = self.down2(p1, t_emb) # 14x14
        p2 = self.pool(d2)         # 7x7
        
        # Down 3
        d3 = self.down3(p2, t_emb) # 7x7
        
        # Mid
        m = self.mid(d3, t_emb)    # 7x7
        
        # Up 1
        u1 = self.up1(m)           # 14x14
        u1 = torch.cat([u1, d2], dim=1) # Concat skip connection
        u1 = self.up_block1(u1, t_emb)  # 14x14
        
        # Up 2
        u2 = self.up2(u1)          # 28x28
        u2 = torch.cat([u2, d1], dim=1) # Concat skip connection
        u2 = self.up_block2(u2, t_emb)  # 28x28
        
        # Output
        return self.outc(u2)

# 3. Setup Dataset and DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Crucial: Maps [0, 1] to [-1, 1]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Filter dataset to only include the TARGET_DIGITS
idx_target = torch.isin(mnist_train.targets, torch.tensor(TARGET_DIGITS))
custom_dataset = Subset(mnist_train, torch.where(idx_target)[0])
loader = DataLoader(custom_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. Apply Configuration via your Dataclass
cfg = DiffusionConfig(
    image_size=IMAGE_SIZE,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS, 
    lr=LEARNING_RATE, 
    beta_min=BETA_MIN,
    beta_max=BETA_MAX,
    num_steps=NUM_STEPS,
    num_sample_grid=NUM_GENERATIONS,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Initialize Model and Diffusion Utilities
diffusion_process = Diffusion(
    beta_min=cfg.beta_min, 
    beta_max=cfg.beta_max, 
    num_steps=cfg.num_steps, 
    device=cfg.device
)

# Use the new Medium U-Net
model = UNetMedium(
    in_channels=cfg.channels, 
    base_dim=BASE_DIM, 
    time_dim=TIME_DIM
).to(cfg.device)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# 6. Train the Model
print(f"Starting training on device: {cfg.device}...")
print(f"Dataset size: {len(custom_dataset)} images of digits {TARGET_DIGITS}")

results = train_diffusion_model(
    cfg=cfg,
    diffusion=diffusion_process,
    model=model,
    loader=loader,
    optimizer=optimizer
)
print("Training complete!")

# 7. Generate and display images
print(f"Generating {cfg.num_sample_grid} sample images...")
samples = generate_images(cfg, diffusion_process, model, num_images=cfg.num_sample_grid)

# Plotting the generated images
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    # Re-normalize from [-1, 1] back to [0, 1] for displaying
    img = (samples[i].cpu().squeeze() + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.axis('off')
    
plt.suptitle(f"Generated MNIST Digits {TARGET_DIGITS}")
plt.tight_layout()
plt.show()


In [ ]:
# Gradual denoising visualization (copy-paste into the notebook)
import torch
import matplotlib.pyplot as plt
from math import ceil

# reseeding
# Requires: `cfg`, `diffusion_process`, and `model` in the notebook scope
diff = diffusion_process
diff.channels = cfg.channels
diff.image_size = cfg.image_size
device = torch.device(cfg.device)

model.eval()
n = 1
x = torch.randn(n, diff.channels, diff.image_size, diff.image_size, device=device)

num_steps = diff.num_steps
steps = torch.linspace(1.0, 1e-3, num_steps, device=device)
dt = 1.0 / num_steps

# Select snapshots to visualize
num_snapshots = 9
snapshot_idxs = torch.linspace(0, num_steps - 1, num_snapshots).long().tolist()

snapshots = []
with torch.no_grad():
    for i, t in enumerate(steps):
        t_batch = torch.full((n,), t, device=device)
        beta_t = diff.beta(t_batch)[:, None, None, None]
        score = model(x, t_batch)
        drift = 0.5 * beta_t * x + beta_t * score

        # Deterministic mean-path update for visualization (no stochastic noise)
        #x = x + drift * dt

        # If you prefer to include stochasticity, replace the above line with:
        x = x + drift * dt + torch.sqrt(beta_t * dt) * torch.randn_like(x)

        if i in snapshot_idxs:
            snapshots.append(x[0].cpu().squeeze())

# Plot snapshots in a grid
cols = int(ceil(num_snapshots ** 0.5))
rows = ceil(num_snapshots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
axes = axes.flatten()
for idx in range(rows * cols):
    ax = axes[idx]
    if idx < len(snapshots):
        img = (snapshots[idx] + 1) / 2  # map from [-1,1] to [0,1]
        ax.imshow(img.clamp(0, 1), cmap="gray")
    ax.axis("off")
fig.suptitle("Denoising progression")
plt.tight_layout()
plt.show()